# 4. AG News Classification with a Transformer Encoder

**New compared to notebook 3:** the LSTM is replaced by a stack of 3 transformer encoder layers (self-attention, 5 heads, on the frozen GloVe embeddings).

* Self-attention is order-agnostic, so **sinusoidal positional encodings** (Vaswani et al., 2017) are added to the embeddings to tell the model where each word is.
* Sentences are now padded at the **end**, and a **padding mask** (`True` = padding) stops attention from looking at the padding tokens.
* There is no "last hidden state": the sentence representation is the **mean of the token outputs over the real (non-padding) tokens**.

Everything else (data, GloVe vocabulary, training loop) is as in notebook 3.

In [ ]:
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm

In [ ]:
# Hyperparameters
max_len = 64
vocab_size = None
dim = 100
device = torch.device('cuda' if torch.cuda.is_available()
                      else 'mps' if torch.backends.mps.is_available() else 'cpu')
batch_size = 128
learning_rate = 8e-4
epochs = 5
seed = 2024

np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True

## Load data

In [ ]:
from utils import get_data
X_train, y_train, X_test, y_test = get_data()

## Split data into words, calculate length of each sentence,
## and calculate the statistics of the dataset

In [ ]:
lens = []
for sentence in X_train:
    lens.append(len(sentence.split(' ')))
lens = np.array(lens)
lens.mean(), np.percentile(lens, 95), lens.max()

## Load Embeddings

In [ ]:
file_name = 'models/glove.6B.100d.txt'
# id 0 = padding, id 1 = unknown word (both zero vectors); GloVe words from 2
word2idx, idx2word = {'<pad>': 0, '<unk>': 1}, {0: '<pad>', 1: '<unk>'}
word2vec = [np.zeros(dim), np.zeros(dim)]
with open(file_name, 'r', encoding='utf-8') as f:
    for idx, line in enumerate(f):
        word, vec = line.split(' ', 1)
        vec = np.fromstring(vec, sep=' ')
        word2idx[word] = idx + 2
        idx2word[idx + 2] = word
        word2vec.append(vec)
word2vec = np.stack(word2vec)
vocab_size = word2vec.shape[0]
word2vec.shape

## Preprocess data and create the tensors

In [ ]:
# prepare the data: post-padding plus a padding mask (True = padding)
def tokenize(sentences):
    tokens, masks = [], []
    for sentence in sentences:
        sentence = sentence.split(' ')[:max_len]
        sentence = [word2idx.get(word, 1) for word in sentence if word]
        if not sentence:  # keep at least one token, or attention has nothing to attend to
            sentence = [1]
        masks.append([False] * len(sentence) + [True] * (max_len - len(sentence)))
        tokens.append(sentence + [0] * (max_len - len(sentence)))  # postpad
    return torch.tensor(tokens), torch.tensor(masks)

X_train_tensor, train_mask_tensor = tokenize(X_train)
y_train_tensor = torch.tensor(y_train)
X_test_tensor, test_mask_tensor = tokenize(X_test)
y_test_tensor = torch.tensor(y_test)

In [ ]:
# data loader
train_dataset = torch.utils.data.TensorDataset(
    X_train_tensor, train_mask_tensor, y_train_tensor)
# train_loader is created in the training loop

test_dataset = torch.utils.data.TensorDataset(
    X_test_tensor, test_mask_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

## Create Model

In [ ]:
class PositionalEncoding(nn.Module):
    # PE[pos, 2i] = sin(pos / 10000^(2i/d)),  PE[pos, 2i+1] = cos(pos / 10000^(2i/d))
    def __init__(self, dim, max_len):
        super().__init__()
        pos = torch.arange(max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, dim, 2) * (-np.log(10000.0) / dim))
        pe = torch.zeros(max_len, dim)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe)

    def forward(self, x):  # x: (B, T, dim)
        return x + self.pe[:x.shape[1]]


class Model(nn.Module):
    def __init__(self, vocab_size, hid_dim, out_dim, n_layers=3, n_heads=5):
        super(Model, self).__init__()
        self.embedding = nn.Embedding(vocab_size, hid_dim, padding_idx=0)
        self.embedding.weight.data.copy_(torch.from_numpy(word2vec).float())
        self.embedding.weight.requires_grad = False
        self.pos_enc = PositionalEncoding(hid_dim, max_len)
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=hid_dim, nhead=n_heads, dim_feedforward=hid_dim*4,
                dropout=0.1, batch_first=True)
            for _ in range(n_layers)])
        self.fc = nn.Linear(hid_dim, out_dim)

    def forward(self, x, mask):
        x = self.pos_enc(self.embedding(x))  # (B, T, dim)
        for layer in self.layers:
            x = layer(x, src_key_padding_mask=mask)
        # mean over the real tokens only
        keep = (~mask).unsqueeze(-1).float()  # (B, T, 1)
        x = (x * keep).sum(1) / keep.sum(1)
        return self.fc(x)

In [ ]:
model = Model(vocab_size, dim, 4).to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

## Train

In [ ]:
for epoch in range(epochs):
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True)
    model.train()
    for X_batch, mask_batch, y_batch in tqdm(train_loader):
        X_batch, mask_batch, y_batch =\
            X_batch.to(device), mask_batch.to(device), y_batch.to(device)
        
        logits = model(X_batch, mask_batch)
        log_probs = F.log_softmax(logits, dim=1)
        loss = F.nll_loss(log_probs, y_batch)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    model.eval()
    with torch.no_grad():
        correct = 0
        total = 0
        for X_batch, mask_batch, y_batch in test_loader:
            X_batch, mask_batch, y_batch =\
                X_batch.to(device), mask_batch.to(device), y_batch.to(device)
            
            y_pred = model(X_batch, mask_batch)
            _, predicted = torch.max(y_pred, 1)
            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)
        acc = correct / total
    print(f'Epoch: {epoch+1:02d}, Test Acc: {acc:.4f}')